In [ ]:
import datetime
import os
from functools import partial
from itertools import product
from pathlib import Path
from string import ascii_lowercase

import colormaps
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import xarray as xr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import (
    BoundaryNorm,
    LinearSegmentedColormap,
    Normalize,
    LogNorm,
    hsv_to_rgb,
    rgb_to_hsv,
)
from matplotlib.dates import DateFormatter
from matplotlib.ticker import MaxNLocator
from scipy.stats import gaussian_kde
from tqdm import tqdm

os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

import altair as alt
from jetutils.anyspell import (
    extend_spells,
    get_spells,
    make_daily,
    mask_from_spells_pl,
    subset_around_onset,
)
from jetutils.data import compute_anomalies_pl, standardize, periodic_rolling_pl
from jetutils.definitions import (
    DATADIR,
    RADIUS,
    DUNCANS_REGIONS_NAMES,
    FACTORS,
    FACTORS_UNITS,
    FIGURES,
    MONTH_NAMES,
    PRETTIER_VARNAME,
    UNITS,
    YEARS,
    get_region,
    polars_to_xarray,
    squarify,
)
from jetutils.geospatial import (
    central_diff,
    compute_relative_anom,
    compute_relative_clim,
    compute_relative_sm,
)
from jetutils.jet_finding import (
    average_jet_categories,
    extract_features,
    jet_position_as_da,
    pers_from_cross_catd,
    spells_from_cross_catd_simple,
    to_one_large,
    persistence_expr,
    connected_from_cross,
    track_jets,
    spells_from_cross
)
from jetutils.plots import (
    COLORS,
    COLORS_EXT,
    STYLE_SHEET,
    TEXTWIDTH_IN,
    WERNLI_FLAIR,
    Clusterplot,
    plot_interp,
)
from jetutils.stats import create_bootstrapped_times

alt.data_transformers.enable("vegafusion")

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

basepath = Path(f"{DATADIR}/exp10") 
# interpd things are broken with exp9. How about exp10?

ALL_TIMES = (
    pl.datetime_range(
        start=pl.datetime(1959, 1, 1),
        end=pl.datetime(2023, 1, 1),
        closed="left",
        interval="6h",
        eager=True,
        time_unit="ms",
    )
    .rename("time")
    .to_frame()
)
summer_filter = ALL_TIMES.filter(pl.col("time").dt.month().is_in([6, 7, 8, 9])).filter(
    pl.col("time").dt.ordinal_day() > 166
)
summer = summer_filter["time"]
summer_daily = summer.filter(summer.dt.hour() == 0)
big_summer = ALL_TIMES.filter(pl.col("time").dt.month().is_in([6, 7, 8, 9]))
big_summer_daily = big_summer.filter(big_summer["time"].dt.hour() == 0)

In [ ]:
jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
props = pl.read_parquet(basepath.joinpath("props.parquet"))
props_catd = squarify(average_jet_categories(props), ["time", "jet"])
props_catd = props_catd.join(
    props_catd.rolling("time", period="2d", group_by="jet").agg(
        **{
            f"{col}_var": pl.col(col).var()
            for col in ["mean_lon", "mean_lat", "mean_s", "s_star"]
        }
    ),
    on=["time", "jet"],
)
props_catd_summer = summer_filter.join(props_catd, on="time")
cross = track_jets(jets, catd=False)
cross_, summary_comp = connected_from_cross(jets, cross, dist_thresh=1.1e6, dis_polar_thresh=0.05)
summary_comp = summary_comp.join(props, on=["time", "jet ID"], suffix="_huh")
cross = cross.join(props, on=["time", "jet ID"], suffix="_huh")
pers = cross.with_columns(pers=persistence_expr()).group_by("time", "jet ID", maintain_order=True).agg(pl.col("pers").max(), pl.col("is_polar_huh").first())

spells_list = spells_from_cross(
    jets,
    cross,
    1.6e6,
    0.4,
    0.1,
    n_STJ=30,
    n_EDJ=30,
    season=summer,
)


# random easy stuff

In [ ]:
plt.style.use(["seaborn-v0_8-darkgrid", STYLE_SHEET])
colors = [COLORS[2], COLORS[1]]

fig, ax = plt.subplots(figsize=(1, 0.5), dpi=100)
ax.axis("off")
for spell_name, color in zip(spells_list, colors):
    ax.plot([], [], color=color, lw=3, label=spell_name)
ax.legend()
plt.show()
min_summer, max_summer = (
    summer.dt.ordinal_day().unique().first(),
    summer.dt.ordinal_day().unique().last(),
)
fig, axes = plt.subplots(
    8,
    8,
    figsize=(9, 4),
    gridspec_kw=dict(wspace=0.05, hspace=0.05, left=0, right=1, bottom=0, top=1),
    subplot_kw=dict(
        xticks=[],
        yticks=[],
        xlim=[-1, max_summer - min_summer],
        ylim=[-1, len(colors) + 1],
    ),
)
axes = axes.ravel()
for ax, year in zip(axes, YEARS):
    ax.text(70, len(spells_list) - 0.3, f"{year}", fontsize=10)
    for j, (name_, spell) in enumerate(spells_list.items()):
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=colors[j],
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = spell.filter(pl.col("time").dt.year() == year)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=colors[j], lw=3)
plt.show()
plt.style.use(["default", STYLE_SHEET])
# fig.savefig(f"{FIGURES}/Persistence/when_spells.pdf")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 4), gridspec_kw=dict(wspace=0, hspace=0))

ax = axes[0]
colors = [COLORS[2], COLORS[1]]
keys = list(spells_list)
i = 1
otheralpha = 0.5
dy = 30
for key, val in spells_list.items():
    huh = (
        summer.dt.ordinal_day()
        .unique()
        .sort()
        .to_frame()
        .with_columns(
            time_2=pl.datetime(year=1959, month=1, day=1) + pl.duration(days="time")
        )
    )
    to_plot = gaussian_kde(val["time"].dt.ordinal_day(), bw_method=0.08).evaluate(
        huh["time"]
    )
    ax.fill_between(
        huh["time_2"],
        i,
        i + dy * to_plot,
        color=colors[1 - i],
        facecolor="none",
        alpha=1.0,
        lw=2,
    )
    ax.fill_between(
        huh["time_2"],
        i,
        i + dy * to_plot,
        color=colors[1 - i],
        alpha=otheralpha,
    )
    i = i - 1
ax.xaxis.set_major_formatter(DateFormatter("%m-%d"))
ax.xaxis.set_tick_params(labelsize=11, rotation=30)
ticks = ax.get_xticks()
ticklabels = ax.get_xticklabels()
ax.set_xticks(ticks, labels=[t.get_text() for t in ticklabels], ha="right")
ax.set_yticks([0.35, 1.5], labels=keys[::-1])
ax.yaxis.set_tick_params(length=0)
ax.set_title("a) Episode distribution")
ylim = ax.get_ylim()
ax.set_ylim(ylim[0], 2 - ylim[0])

ax = axes[1]
dy = 1.2
bins = np.logspace(-5.0, -0.1, 101)
summer_pers = summer_filter.join(pers, on="time")
y1 = summer_pers.filter(pl.col("is_polar_huh") < 0.4)["pers"]
y2 = summer_pers.filter(pl.col("is_polar_huh") > 0.6)["pers"]
for i, y in enumerate([y1, y2]):
    y_smo = gaussian_kde(
        np.log10(y.replace((0., float("inf")), None).drop_nulls()), bw_method=0.2
    ).evaluate(np.log10(bins))
    y_smo = (1 - i) + dy * y_smo
    ax.fill_between(bins, (1 - i), y_smo, color=COLORS[2 - i], alpha=otheralpha)
    ax.plot(bins, y_smo, color=COLORS[2 - i], lw=2)
ax.set_xscale("log")
ax.set_yticks([])
ax.set_title("b) Persistence [s/m]")
ylim = ax.get_ylim()
ax.set_ylim(ylim[0], 2 - ylim[0])

ax = axes[2]
dy = 2
bins = np.arange(5, 20, 0.15) - 0.125
for i, jet in enumerate(["EDJ", "STJ"]):
    y = spells_list[jet].group_by("spell").agg(pl.col("len").first())["len"] / 4
    y_smo = gaussian_kde(y, bw_method=0.2).evaluate(bins)
    y_smo = i + dy * y_smo
    ax.fill_between(bins, i, y_smo, color=COLORS[1 + i], alpha=otheralpha)
    ax.plot(bins, y_smo, color=COLORS[1 + i], lw=2)
ax.set_yticks([0, 0.5, 1, 1.5], labels=[str(i) for i in [0, 0.5] * 2])
ax.tick_params("y", left=False, right=True, labelleft=False, labelright=True)
ax.set_title("c) Episode length [day]")
ylim = ax.get_ylim()
ax.set_ylim(ylim[0], 2 - ylim[0])
# fig.savefig(f"{FIGURES}/Persistence/stats.pdf")

In [ ]:
wr_names = ["No", "GLBl", "ScTr", "EuBl", "ScBl"]
colors = ["#6C6C6C", "#7E7EF4", "#F2A685", "#8FC386", "green"]

grams_wr = pl.read_parquet(f"{DATADIR}/grams_wr.parquet")
grams_wr = grams_wr.with_columns(
    **{
        f"winner_{i}": pl.when(pl.col("winner") == i)
        .then(pl.lit(1.0))
        .otherwise(pl.lit(0.0))
        for i in range(5)
    }
)
width = 0.25
fig, axes = plt.subplot_mosaic(
    [["a)", "b)", "c)"], ["a)", "b)", "d)"]],
    figsize=(8, 4),
    constrained_layout=True,
    sharey=True,
    width_ratios=[1, 1, 0.5],
    gridspec_kw=dict(wspace=0.1),
)
for i, (spell_of, ax) in enumerate(zip(["STJ", "EDJ"], list(axes.values()))):
    grams_wr_masked = mask_from_spells_pl(
        spells_list[spell_of], grams_wr, time_before=datetime.timedelta(days=3)
    )
    huh = grams_wr_masked.group_by(["relative_index"]).mean().sort("relative_index")
    alive_spells = (
        grams_wr_masked.group_by("relative_index")
        .agg(pl.col("spell").n_unique())
        .sort("relative_index")["spell"]
        .to_numpy()
    )
    x = huh["relative_index"].to_numpy() / 4
    filter_ = alive_spells > 3
    x = x[filter_]
    bottom = np.zeros(len(x))
    for j in [*np.arange(1, 5), 0]:
        height = huh[f"winner_{j}"].to_numpy()[filter_]
        ax.bar(
            x,
            height,
            bottom=bottom,
            facecolor=colors[j],
            width=width,
            label=wr_names[j],
        )
        bottom = bottom + height
    ax.set_xlabel("Relative time around onset [day]")
    ax.set_title(
        f"{ascii_lowercase[i]}) {alive_spells.max():n} episodes of the {spell_of[:3]}"
    )
    ax.set_xlim(x[0] - width / 2, x[-1] + width / 2)
    ybounds = [1, 1.05]
    im = ax.pcolormesh(
        x,
        ybounds,
        alive_spells[filter_][None, 1:],
        zorder=2,
        cmap=colormaps.greys,
        alpha=0.7,
        vmin=0,
    )
    ax.axhline(1, color="black")
    ax.vlines(0, 0, 1, color="black", ls="dotted", lw=1.2)
h, l = axes["b)"].get_legend_handles_labels()
axes["c)"].set_axis_off()
axes["c)"].legend(h, l, fontsize=12, ncol=1, loc="upper left", title="WR names")
axes["a)"].set_ylabel("WR frequency")
ax = axes["d)"]
monthly_means = (
    grams_wr.filter(pl.col("time").dt.month() > 5)
    .group_by(pl.col("time").dt.month())
    .agg(*[pl.col(f"winner_{i}").mean() for i in range(5)])
    .sort("time")
)
x = np.array([6, 7, 8, 9])
bottom = np.zeros(len(x))
for j in [*np.arange(1, 5), 0]:
    height = monthly_means[f"winner_{j}"].to_numpy()
    ax.bar(x, height, bottom=bottom, facecolor=colors[j], width=0.9, label=wr_names[j])
    bottom = bottom + height
ax.set_xticks(x, labels="JJAS")
ax.set_title("c) Monthly means")
# fig.savefig(f"{FIGURES}/Persistence/wrs_bars.pdf")

In [ ]:
from string import ascii_lowercase

from scipy.stats import chi2, norm, t

data_vars = [
    "mean_lon",
    "mean_lat",
    "mean_theta",
    "mean_lev",
    "mean_s",
    "width",
    "tilt",
    "waviness1",
    "waviness2",
    "wavinessDC16",
    "wavinessFV15",
    "mean_lat_var",
    "mean_s_var",
    "is_polar",
    "int",
    "int_over_europe",
]
mode_dict = {
    "": "Connected components",
    "catd": "Full distance",
    "com": "COM distance",
}


def func(col):
    if ":" in col and col.split(":")[-1] == "var":
        return pl.col(col.split(":")[0]).var()
    return pl.col(col).mean()


def mean_confidence(col: pl.Series, q: float) -> pl.Series:
    n = (col.is_not_null() & col.is_not_nan()).sum()
    mu = col.mean()
    s_sq = col.var()
    if s_sq is None:
        return None
    s = np.sqrt(s_sq)
    sign = 1 - 2 * int(q < 0.5)
    q = q if q < 0.5 else 1 - q
    if n > 10:
        to_ret = mu + sign * np.abs(norm.ppf(q=q)) / np.sqrt(n) * s
    else:
        to_ret = mu + sign * s / np.sqrt(n) * t.ppf(q=1 - q, df=n - 1)
    to_ret = np.clip(to_ret, mu - 5 * s, mu + 5 * s)
    return to_ret


def var_confidence(col: pl.Series, q) -> float:
    n = (col.is_not_null() & col.is_not_nan()).sum()
    s_sq = col.var()
    if s_sq is None:
        return None
    sign = 1 - 2 * int(q < 0.5)
    if n > 50:
        q = q if q < 0.5 else 1 - q
        to_ret = s_sq + sign * np.sqrt(2 / n) * np.abs(norm.ppf(q)) * s_sq
    else:
        to_ret = (n - 1) * s_sq / chi2.ppf(1 - q, df=n - 1)
    return np.clip(to_ret, 0, s_sq * 2)


def func_q(col, q):
    if ":" in col and col.split(":")[-1] == "var":
        return pl.col(col.split(":")[0]).map_batches(
            partial(var_confidence, q=q), return_dtype=pl.Float64(), returns_scalar=True
        )
    return pl.col(col).map_batches(
        partial(mean_confidence, q=q), return_dtype=pl.Float64(), returns_scalar=True
    )


q_mean = 1e-15
for spell_of in ["STJ", "EDJ"]:
    spells_from_jet = spells_list[spell_of]
    n_spells = spells_from_jet["spell"].n_unique()
    props_masked = mask_from_spells_pl(
        spells_from_jet, props_catd, time_before=datetime.timedelta(days=4)
    )
    props_masked = props_masked.filter(
        pl.col("spell").n_unique().over("relative_index") > 12
    )
    aggs = {col: func(col) for col in data_vars}
    aggs = aggs | {f"{col}_10": func_q(col, q_mean) for col in data_vars}
    aggs = aggs | {f"{col}_90": func_q(col, 1 - q_mean) for col in data_vars}
    explode_list = [f"{col}_10" for col in data_vars] + [
        f"{col}_90" for col in data_vars
    ]
    aggs = aggs | {"alive": pl.col("time").len()}
    mean_ps = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(
        **aggs
    )
    aggs_ = {col: func_q(col, 0.95) for col in data_vars}
    q25 = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(
        **aggs_
    )
    aggs_ = {col: func_q(col, 0.05) for col in data_vars}
    q75 = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(
        **aggs_
    )
    fig, axes = plt.subplots(4, 4, figsize=(11, 11), tight_layout=False, sharex="all")
    axes = axes.ravel()
    means = props_catd_summer.group_by("jet", maintain_order=True).agg(**aggs)
    alive_spells = (
        props_masked.group_by("relative_index")
        .agg(pl.col("spell").n_unique())
        .sort("relative_index")["spell"]
        .to_numpy()
    )
    for j, jet in enumerate(["STJ", "EDJ"]):
        to_plot = mean_ps.filter(pl.col("jet") == jet)
        q25_ = q25.filter(pl.col("jet") == jet)
        q75_ = q75.filter(pl.col("jet") == jet)
        x = to_plot["relative_index"].unique().to_numpy() / 4
        for ax, data_var, letter in zip(axes, data_vars, ascii_lowercase):
            factor = 1e9 if data_var in ["int_over_europe", "int"] else 1
            factor = 1e5 if data_var == "width" else factor
            ax.plot(x, to_plot[data_var] / factor, color=COLORS[2 - j], lw=2.5)
            ax.fill_between(
                x,
                q25_[data_var] / factor,
                q75_[data_var] / factor,
                color=COLORS[2 - j],
                alpha=0.4,
            )
            mean = means.filter(pl.col("jet") == jet)[data_var].item() / factor
            q10 = means.filter(pl.col("jet") == jet)[f"{data_var}_10"].item() / factor
            q90 = means.filter(pl.col("jet") == jet)[f"{data_var}_90"].item() / factor
            ax.plot([x[0], x[-1]], [mean, mean], color=COLORS[2 - j], ls="dashed", lw=2)
            if j == 0:
                factor_str = (
                    "" if factor == 1 else rf"$10^{int(np.log10(factor))} \times $"
                )
                ax.set_title(
                    rf"{letter}) {PRETTIER_VARNAME.get(data_var, data_var)} [{factor_str}{UNITS.get(data_var, '~')}]"
                )
            ax.yaxis.set_major_locator(MaxNLocator(4, integer=True))
    for i, ax in enumerate(axes):
        ax.axvline(0, zorder=1, color="black", lw=2)
        if i > 11:
            ax.set_xlabel("Relative time around onset [days]", color="black")
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        ybounds = [
            ylim[0] - 0.05 * (ylim[1] - ylim[0]),
            ylim[0] + 0.05 * (ylim[1] - ylim[0]),
        ]
        # im = ax.pcolormesh(
        #     x,
        #     ybounds,
        #     alive_spells[None, :-1],
        #     zorder=-10,
        #     cmap=colormaps.greys,
        #     alpha=0.7,
        #     vmin=0,
        # )
    fig.set_constrained_layout(True)
    mode = mode_dict[spell_of[4:]]
    fig.suptitle(
        f"Persistent episodes of the {spell_of[:3]}. Mode: {mode}. {props_masked['spell'].n_unique()} spells"
    )
    # fig.savefig(f"{FIGURES}/Persistence/{spell_of}_props_{n_spells}spells.pdf")
    # plt.close()

In [ ]:
clusters_da = np.abs(xr.open_dataarray(basepath.joinpath("cluster_df.nc")).load())
clusters_da = clusters_da.interp(lat=np.arange(32, 72, 0.5), method="nearest")

# plt.savefig(f"{FIGURES}/jet_persistence/regions.png")

region_T = compute_anomalies_pl(
    pl.read_parquet(basepath.joinpath("region_T_6H.parquet")), ["region"]
)
region_T = region_T.rolling("time", period="3d", group_by="region").agg(
    pl.col("t2m").mean()
)
hws = get_spells(
    region_T,
    pl.col("t2m") > pl.col("t2m").quantile(0.95),
    group_by=["region"],
    fill_holes=datetime.timedelta(hours=18),
    minlen=datetime.timedelta(days=3),
).sort("region")
region_tp = compute_anomalies_pl(
    pl.read_parquet(basepath.joinpath("region_tp_6H.parquet")), ["region"]
)
region_tp = region_tp.rolling("time", period="3d", group_by="region").agg(
    pl.col("tp").mean()
)
pes = get_spells(
    region_tp,
    pl.col("tp") > pl.col("tp").quantile(0.95),
    group_by=["region"],
    fill_holes=datetime.timedelta(hours=6),
    minlen=datetime.timedelta(days=3),
).sort("region")
drs = get_spells(
    region_tp,
    pl.col("tp") < pl.col("tp").quantile(0.05),
    group_by=["region"],
    fill_holes=datetime.timedelta(hours=6),
    minlen=datetime.timedelta(days=3),
).sort("region")

colors_regions = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]
figure = plt.figure(figsize=(10, 4.5), constrained_layout=True)
subfigs = figure.subfigures(1, 2, wspace=0.00, width_ratios=[6.5, 3.5])
with plt.style.context(["seaborn-v0_8-darkgrid", STYLE_SHEET]):
    fig = subfigs[0]
    axes = fig.subplots(2, 2, sharex="col", sharey="row")
    axes = axes.T
    for i, spell_of in enumerate(["STJ", "EDJ"]):
        spells = extend_spells(spells_list[spell_of])
        for j, (name, df) in enumerate({"t2m": region_T, "tp": region_tp}.items()):
            ax = axes[i, j]
            region_ts_masked = mask_from_spells_pl(
                spells, df, time_before=datetime.timedelta(days=4)
            )
            region_ts_masked = subset_around_onset(
                region_ts_masked, around_onset=datetime.timedelta(days=13)
            )
            x, alive_spells = (
                region_ts_masked.group_by(["region", "relative_index"])
                .agg(pl.col("spell").n_unique())
                .sort("relative_index")
                .filter(pl.col("region") == 1)
                .to_numpy()
                .T[1:]
            )
            filter_ = alive_spells > 3
            huh = (
                region_ts_masked.group_by(["region", "relative_index"])
                .mean()
                .sort(["region", "relative_index"])
            )
            ax.axhline(0, color="black")
            ax.axvline(0, color="black")
            for reg, huh_ in huh.group_by("region", maintain_order=True):
                y = huh_[name].to_numpy()
                y = y * 1000 if name == "tp" else y
                ax.plot(
                    x[filter_],
                    y[filter_],
                    color=colors_regions[reg[0] - 1],
                    label=DUNCANS_REGIONS_NAMES[reg[0] - 1],
                    lw=3,
                )
            k = 2 * j + i
            ax.annotate(
                f"{ascii_lowercase[k]})",
                xy=(0, 1),
                xycoords="axes fraction",
                xytext=(+0.5, -0.5 - 6.5 * float(k == 0)),
                textcoords="offset fontsize",
                fontsize="medium",
                verticalalignment="top",
                fontfamily="serif",
                bbox=dict(facecolor="0.7", edgecolor="none", pad=3.0),
            )
    # axes[0, 0].legend(ncol=2, fontsize=13.5, labelspacing=0.3, markerscale=.1, columnspacing=.6, facecolor="white", framealpha=.5, fancybox=True, frameon=True)
    axes[0, 0].set_ylabel(PRETTIER_VARNAME["t2m"])
    axes[0, 1].set_ylabel(PRETTIER_VARNAME["tp"])
    axes[0, 0].set_title("STJ episodes")
    axes[1, 0].set_title("EDJ episodes")
    axes[0, 1].set_xlabel("Time around onset [day]")
    axes[1, 1].set_xlabel("Time around onset [day]")
    ylim = axes[0, 1].get_ylim()
    for i, spell_of in enumerate(["STJ", "EDJ"]):
        spells = extend_spells(
            spells_list[spell_of], time_before=datetime.timedelta(days=4)
        )
        spells = subset_around_onset(spells, around_onset=datetime.timedelta(days=13))
        x, alive_spells = (
            spells.group_by("relative_index")
            .agg(pl.col("spell").n_unique())
            .sort("relative_index")
            .to_numpy()
            .T
        )
        filter_ = alive_spells > 3
        ax = axes[i, 1]
        ybounds = [
            ylim[0] - 0.05 * (ylim[1] - ylim[0]),
            ylim[0] + 0.05 * (ylim[1] - ylim[0]),
        ]
        im = ax.pcolormesh(
            x[filter_],
            ybounds,
            alive_spells[None, filter_][:, :-1],
            zorder=-10,
            cmap=colormaps.greys,
            alpha=0.7,
            vmin=0,
        )
    axes[0, 1].set_ylim(ybounds[0])


clu = Clusterplot(1, 1, (-10, 40, 35, 72), row_height=5, fig=subfigs[1])
cmap = colormaps.pastel
ax = clu.axes[0]
unique_clusters = np.arange(1, 7)
norm = BoundaryNorm(np.arange(cmap.N) + 0.5, cmap.N)
clusters_da.assign_coords(lon=clusters_da.lon - clu.central_longitude).plot(
    ax=ax, cmap=cmap, norm=norm, add_colorbar=False, add_labels=False
)
for j in range(6):
    lo = (
        clusters_da.lon.where(clusters_da == (j + 1)).mean().item()
        - j
        - 3 * (j == 0)
        + 2 * (j == 2)
        + 3 * (j == 1)
        - (j == 4)
        - clu.central_longitude
    )
    la = clusters_da.lat.where(clusters_da == (j + 1)).mean().item() - (j == 5) * 2
    color = "black"
    ax.text(
        lo,
        la,
        DUNCANS_REGIONS_NAMES[j],
        ha="center",
        va="center",
        fontweight="bold",
        color=color,
        fontsize=12,
    )

ax.annotate(
    f"{ascii_lowercase[k + 1]})",
    xy=(float(k == 0), 1),
    xycoords="axes fraction",
    xytext=(+0.5 - 2 * float(k == 0), -0.5),
    textcoords="offset fontsize",
    fontsize="medium",
    verticalalignment="top",
    fontfamily="serif",
    bbox=dict(facecolor="0.7", edgecolor="none", pad=3.0),
)
# figure.savefig(f"{FIGURES}/Persistence/region_timeseries_30spells.pdf")

# interp

In [ ]:
all_plot_kwargs = {
    "theta:clim": [8, colormaps.bilbao_r, [330, 370]],
    "theta:anom": [8, colormaps.BlWhRe, [-6, 6]],
    "theta:clim_grad": [8, colormaps.bilbao_r, [0, 40]],
    "theta:anom_grad": [10, colormaps.BlWhRe, [-12, 12]],
    "PV:clim": [9, WERNLI_FLAIR, [0, 10]],
    "PV:anom": [8, colormaps.BlWhRe, [-1.2, 1.2]],
    "PV:clim_grad": [10, colormaps.bilbao_r, [0, 10]],
    "PV:anom_grad": [10, colormaps.BlWhRe, [-4, 4]],
    "EKE250:clim": [9, colormaps.cet_l_bmy_r, [0, 200]],
    "EKE250:anom": [9, colormaps.BlWhRe, [-30, 30]],
    "hor:clim": [12, colormaps.BlWhRe, [-12, 12]],
    "hor:anom": [12, colormaps.BlWhRe, [-12, 12]],
    "hor:abs": [12, colormaps.BlWhRe, [-12, 12]],
    "t2m:clim": [6, colormaps.bilbao_r, [280, 300]],
    "t2m:anom": [8, colormaps.BlWhRe, [-4, 4]],
    "tp:clim": [9, colormaps.freeze_r, [0, 6]],
    "tp:anom": [9, colormaps.brbg, [-2, 2]],
}

for varname in ["APVO", "CPVO"]:
    all_plot_kwargs[f"{varname}:clim"] = [7, colormaps.cet_l_bmy_r, [0, 70]]
    all_plot_kwargs[f"{varname}:anom"] = [8, colormaps.BlWhRe, [-16, 16]]
    all_plot_kwargs[f"{varname}:abs"] = [8, colormaps.BlWhRe, [0, 70]]

variables = list(all_plot_kwargs)

summer_doy = summer_daily.dt.ordinal_day().unique()

def do_one(varname, basepath, jet, spells, summer, n_bootstraps, factor):
    varname, mode = varname.split(":")
    varname_no_number = varname.rstrip("0123456789")
    varname_ = f"{varname}_interp"
    grad = mode[-4:] == "grad"
    is_polar = jet == "EDJ"
    jet_id = int(is_polar)

    df = pl.scan_parquet(basepath.joinpath(f"{varname}_relative.parquet"))
    if varname_ not in df.collect_schema().names():
        print(varname_)
        if f"{varname_no_number}_interp" in df.collect_schema().names():
            df = df.rename({f"{varname_no_number}_interp": varname_})
        else:
            df = df.rename({"vort_interp": varname_})
    if "jet ID" not in df.columns:
        df = df.with_columns(**{"jet ID": pl.col("is_polar").cast(pl.UInt32())})
    grad_expr = (
        (central_diff(pl.col(varname_).sort_by("n")) / central_diff(pl.col("n").sort()))
        * 1e6
    ).abs()
    if grad:
        df = df.with_columns(
            **{varname_: grad_expr.over("norm_index", "time", "jet ID")}
        )
    clim = compute_relative_clim(df, varname)
    if mode in ["clim", "clim_grad"]:
        to_plot = compute_relative_sm(clim, varname, summer_doy)
        to_plot = to_plot.filter(pl.col("dayofyear") == 1, pl.col("jet ID") == jet_id)
        to_plot = to_plot.drop("jet ID", "dayofyear")
        pvals = None
    elif mode in ["anom", "anom_grad"]:
        winsize = 31
        halfwinsize = int(
            winsize / 2
        )  # TODO: make data.periodic_rolling_pl work with lazyframes. UPDATE: looks really hard
        clim = clim.rolling(
            pl.col("dayofyear").cast(pl.UInt32()),
            period=f"{winsize}i",
            offset=f"-{halfwinsize + 1}i",
            group_by=["jet ID", "norm_index", "n"],
        ).agg(pl.col(varname_).mean())
        to_plot = compute_relative_anom(df, varname, clim)
        ts_bootstrapped = create_bootstrapped_times(spells, summer, n_bootstraps).lazy()
        to_plot = (
            ts_bootstrapped.join(to_plot, on="time")
            .filter(pl.col("jet ID") == jet_id)
            .sort("sample_index", "spell", "inside_index", "norm_index", "n")
        )
        to_plot = to_plot.group_by(
            "sample_index", "norm_index", "n", maintain_order=True
        ).agg(pl.col(varname_).mean())
        pvals = (
            to_plot.group_by("norm_index", "n", maintain_order=True)
            .agg((pl.col(varname_).rank().last() - 1) / n_bootstraps)
            .sort("norm_index", "n")
        )
        pvals = pvals.with_columns(
            **{varname_: 2 * pl.min_horizontal(pl.col(varname_), 1 - pl.col(varname_))}
        )
        to_plot = (
            to_plot.filter(pl.col("sample_index") == n_bootstraps)
            .drop("sample_index")
            .sort("norm_index", "n")
        )
    else:
        to_plot = spells.lazy().join(df.filter(pl.col("jet ID") == jet_id), on="time")
        to_plot = (
            to_plot.group_by("norm_index", "n")
            .agg(pl.col(varname_).mean())
            .sort("norm_index", "n")
        )
        pvals = None
    to_plot = to_plot.with_columns(pl.col(varname_) * factor)
    to_plot = to_plot.collect()
    to_plot = polars_to_xarray(to_plot, ["norm_index", "n"]).T
    if pvals is not None:
        pvals = pvals.collect()
        pvals = polars_to_xarray(pvals, ["norm_index", "n"]).T
    return to_plot, pvals

n_bootstraps = 40
for jet in ["STJ", "EDJ"]:
    spells = spells_list[jet]
    nspells = spells["spell"].n_unique()
    # spells = spells.filter(*during_filter)
    for varname in tqdm(variables):
        varname_, rest = varname.split(":")
        if varname_ == "PV":
            varname = f"PV330:{rest}" if jet == "EDJ" else f"PV350:{rest}"
        factor = FACTORS_UNITS.get(varname_.rstrip("035"), 1)
        ofile = Path(f"{FIGURES}/Persistence/figure_data_graphspells2/{jet}_{varname}.nc")
        ofile_pvals = Path(
            f"{FIGURES}/Persistence/figure_data_graphspells2/{jet}_{varname}_pvals.nc"
        )
        if ofile.is_file():
            continue
        to_plot, pvals = do_one(
            varname, basepath, jet, spells, summer, n_bootstraps, factor
        )
        to_plot.to_netcdf(ofile)
        if pvals is not None:
            pvals.to_netcdf(ofile_pvals)

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data_graphspells2")
odir = Path(f"{FIGURES}/Persistence/RWB")
odir.mkdir(exist_ok=True)
variables = {}

for varname in ["APVO", "CPVO"]:
    variables[f"{varname}:clim"] = [8, colormaps.cet_l_bmy_r, [0, 70]]
    # variables[f"{varname}:abs"] = [8, colormaps.cet_l_bmy_r, [0, 70]]
    variables[f"{varname}:anom"] = [8, colormaps.BlWhRe, [-16, 16]]

for jet in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet,
        handle_pvals="hatch",
        n_col=2,
        square_len=3.3,
        transpose=True,
    )
    # fig.savefig(odir.joinpath(f"{jet}.pdf"))
    # plt.close()

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data_graphspells2")
odir = Path(f"{FIGURES}/Persistence/upper_level")
odir.mkdir(exist_ok=True)

variables = {
    "theta:clim": [8, colormaps.bilbao_r, [330, 370]],
    "theta:anom": [8, colormaps.BlWhRe, [-6, 6]],
    "theta:clim_grad": [8, colormaps.bilbao_r, [0, 40]],
    "theta:anom_grad": [10, colormaps.BlWhRe, [-12, 12]],
    "PV:clim": [9, WERNLI_FLAIR, [0, 10]],
    "PV:anom": [8, colormaps.BlWhRe, [-1.2, 1.2]],
    "PV:clim_grad": [10, colormaps.bilbao_r, [0, 10]],
    "PV:anom_grad": [10, colormaps.BlWhRe, [-4, 4]],
}
for jet in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet,
        handle_pvals="hatch",
        n_col=2,
        square_len=3.3,
        transpose=True,
    )
    # fig.savefig(odir.joinpath(f"{jet}.pdf"))
    # plt.close()

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data_graphspells2")
odir = Path(f"{FIGURES}/Persistence/ground")
odir.mkdir(exist_ok=True)
variables = {
    "t2m:clim": [6, colormaps.bilbao_r, [280, 300]],
    "t2m:anom": [8, colormaps.BlWhRe, [-4, 4]],
    "tp:clim": [9, colormaps.freeze_r, [0, 6]],
    "tp:anom": [9, colormaps.brbg, [-2, 2]],
}

for jet in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet,
        handle_pvals="hatch",
        n_col=2,
        square_len=3.3,
        transpose=True,
    )
    # fig.savefig(odir.joinpath(f"{jet}.pdf"))
    # plt.close()

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data_graphspells2")
odir = Path(f"{FIGURES}/Persistence/eddies")
odir.mkdir(exist_ok=True)

variables = {
    "EKE250:clim": [9, colormaps.cet_l_bmy_r, [0, 200]],
    "EKE250:anom": [9, colormaps.BlWhRe, [-30, 30]],
    "hor:clim": [12, colormaps.BlWhRe, [-4, 4]],
    "hor:anom": [12, colormaps.BlWhRe, [-4, 4]],
}
for jet in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet,
        handle_pvals="hatch",
        n_col=2,
        square_len=3.3,
        transpose=True,
    )
    # fig.savefig(odir.joinpath(f"{jet}.pdf"))
    # plt.close()

# Fing around

In [ ]:
I_want = ["dayofyear", "mean_lon", "mean_lat", "mean_s", "mean_theta", "wavinessR16", "wavinessDC16", "tilt", "is_polar_huh", "dist", "dis_polar", "ds", "dtheta"]
def _expr(col):
    if col != "dayofyear":
        return pl.col(col).mean()
    col = pl.col(col)
    factor = 365 / np.pi
    col = col / factor
    col = pl.arctan2(col.sin().mean(), col.cos().mean()) * factor
    return col

I_want_exprs = [_expr(col) for col in I_want]
n_cuts = 301

long_ones = summary_comp.filter(pl.col("len") > pl.col("len").first().over("spell").quantile(0.996), )
fig, axes = plt.subplots(2, 3, figsize=(10, 5), constrained_layout=True)
axes = axes.ravel()
things = ["mean_lon", "mean_lat", "mean_lev", "mean_theta", "wavinessDC16", "is_polar_huh"]
for spell, long_one in long_ones.group_by("spell"):
    for ax, thing in zip(axes, things):
        ax.plot(np.arange(long_one.shape[0]), long_one[thing])
        
long_ones = summary_comp.filter(pl.col("len") > pl.col("len").first().over("spell").quantile(0.94), pl.col("is_polar").mean().over("spell") > 0.64)
fig, axes = plt.subplots(2, 3, figsize=(10, 5), constrained_layout=True)
axes = axes.ravel()
things = ["mean_lon", "mean_lat", "mean_lev", "mean_theta", "wavinessDC16", "is_polar_huh"]
for spell, long_one in long_ones.group_by("spell"):
    for ax, thing in zip(axes, things):
        ax.plot(np.arange(long_one.shape[0]), long_one[thing])

In [ ]:
mixed = summary_comp.filter(
    pl.col("is_polar_huh").diff().sum().abs().over("spell") > 0.3,
    pl.col("len") > 3
)
fig, axes = plt.subplots(2, 2)
axes = axes.ravel()
things = ["mean_lon", "mean_lat", "mean_theta", "is_polar_huh"]
for spell, long_one in mixed.group_by("spell"):
    for ax, thing in zip(axes, things):
        ax.plot(np.arange(long_one.shape[0]), long_one[thing])

In [ ]:
_ = plt.hist(summary_comp.group_by("spell").agg(pl.col("is_polar_huh").mean())["is_polar_huh"], bins=61)

In [ ]:
int_ = jets.group_by("time", "jet ID").agg(pl.col("int").first(), pl.col("is_polar").mean())
int1 = jets.filter(pl.col("is_polar") < 0.5)["int"]
int2 = jets.filter(pl.col("is_polar") > 0.5)["int"]
bins = np.linspace(7.9, 9.1, 21)
plt.hist(np.log10(int2), bins=bins, alpha=0.5)
plt.hist(np.log10(int1), bins=bins, alpha=0.5)
plt.axvline(np.log10(1.4e8), color="black")